# Eager vs Lazy

pandas always runs **eagerly**: each line executes the moment you run it. Polars can do that too, but it also has a **lazy** mode: you describe the whole query first, Polars builds a plan, optimizes it, and only runs it when you call `.collect()`.

Same question as on the slides: **how many delivered orders were placed each year?**

Docs: [docs.pola.rs/user-guide/concepts/lazy-api](https://docs.pola.rs/user-guide/concepts/lazy-api/)

In [1]:
import polars as pl

path = "../data/olist_orders.csv.gz"

## Eager: `read_csv`

`read_csv` loads **all 8 columns and all 99,441 rows** into memory right away. Each following step then works on the full DataFrame.

In [2]:
df = pl.read_csv(path, try_parse_dates=True)

(
    df
    .filter(pl.col("order_status") == "delivered")
    .group_by(pl.col("order_purchase_timestamp").dt.year().alias("year"))
    .len()
    .sort("year")
)

year,len
i32,u32
2016,267
2017,43428
2018,52783


## Lazy: `scan_csv`

Swap `read_csv` for `scan_csv` and nothing is read yet: `q` is a **plan**, not data.

In [3]:
q = (
    pl.scan_csv(path, try_parse_dates=True)
    .filter(pl.col("order_status") == "delivered")
    .group_by(pl.col("order_purchase_timestamp").dt.year().alias("year"))
    .len()
    .sort("year")
)

print(q.explain())

SORT BY [col("year")]
  AGGREGATE[maintain_order: false]
    [len()] BY [col("order_purchase_timestamp").dt.year().alias("year")]
    FROM
    simple π 1/1 ["order_purchase_timestamp"]
      Csv SCAN [../data/olist_orders.csv.gz]
      PROJECT 2/8 COLUMNS
      SELECTION: col("order_status") == "delivered"
      ESTIMATED ROWS: 157634


Read the plan from the bottom up:

- `PROJECT 2/8 COLUMNS`: Polars saw the whole query, so it reads only the 2 columns it needs (**projection pushdown**).
- `SELECTION: ...`: the filter runs *while* reading, so rows that aren't delivered are never kept (**predicate pushdown**).

pandas can't do this, because it runs each line without knowing what comes next.

Now run the plan:

In [4]:
q.collect()

year,len
i32,u32
2016,267
2017,43428
2018,52783


## Is it faster?

`%timeit` runs each version several times and reports the average.

In [5]:
%timeit -r 3 -n 5 pl.read_csv(path, try_parse_dates=True).filter(pl.col("order_status") == "delivered").group_by(pl.col("order_purchase_timestamp").dt.year().alias("year")).len()
%timeit -r 3 -n 5 q.collect()

70.5 ms ± 674 μs per loop (mean ± std. dev. of 3 runs, 5 loops each)


40.5 ms ± 687 μs per loop (mean ± std. dev. of 3 runs, 5 loops each)


Your numbers will differ by laptop, but lazy should win even on this small file. Eager parses all 8 columns, including 5 date columns; lazy reads 2 columns and parses only the one date it needs. The bigger the file, and the more columns you don't need, the bigger the gap, in both time and memory.

**Takeaway:** for quick exploration of small data, eager `read_csv` is fine. For big files, or a pipeline you'll run again and again, use `scan_csv` / `scan_parquet` and finish with `.collect()`.